# 📓 Notebook 3｜密度估計：ML/MAP/Parzen/EM

> 對應講義 Part 4。做三件事：ML 參數估計與偏誤、Parzen 平滑實驗、EM 混合估計。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
# ---- ML：估高斯 μ 與 σ²（σ² 的 ML 有偏！）----
mu_true, s2_true = 2.0, 4.0
def ml_est(N, iters=2000):
    mu_hat = []; s2_hat = []
    for _ in range(iters):
        x = rng.normal(mu_true, np.sqrt(s2_true), N)
        mu_hat.append(x.mean()); s2_hat.append(((x - x.mean())**2).mean())
    return np.mean(mu_hat), np.mean(s2_hat)
for N in [5, 20, 100, 500]:
    mu_, s2_ = ml_est(N)
    msg = 'N=%d: E[mu_hat]=%.3f (true 2.0)  E[s2_ML]=%.3f (true 4.0, bias*(N-1)/N -> %.3f)' % (N, mu_, s2_, (N-1)/N*4)
    print(msg)

In [ ]:
# ---- Parzen 窗：h 的抉擇（修好後數值積分對照）----
from scipy.stats import norm
npdf = norm(1, 1)

def parzen(N, h):
    xr = rng.normal(1, 1, N)
    g = np.linspace(-2.5, 4.5, 501)
    k = np.exp(-0.5*((g[:,None]-xr[None,:])/h)**2)/(h*np.sqrt(2*np.pi))
    return g, k.mean(1)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (h, N) in zip(axes, [(0.05, 200), (0.5, 200), (2.0, 200)]):
    g, est = parzen(N, h)
    ax.plot(g, npdf.pdf(g), 'k--', label='真值 N(1,1)')
    ax.plot(g, est, 'r-', label=f'Parzen h={h}, N={N}')
    ax.legend(fontsize=8); ax.set_title(f'h={h}：{"刺蝟(高方差)" if h<0.1 else "平滑(偏差大)" if h>1 else "不錯"}')
plt.tight_layout(); plt.show()

### 附錄｜手刻 EM 於兩高斯混合（課本 Example 2.8 精神）

> 兩分量、未知 (μ1,μ2,σ1²,σ2²,P)，標籤未知 → E/M 迭代，似然單調上升。

In [ ]:
rng = np.random.default_rng(1)
N = 300
lab = rng.random(N) < 0.7
X = np.where(lab, rng.normal(1.0, 0.4, N), rng.normal(4.0, 0.8, N)).reshape(-1, 1)

def gauss1(x, mu, s2): return np.exp(-(x-mu)**2/(2*s2))/np.sqrt(2*np.pi*s2+1e-12)
xx = X.ravel()                                  # 拉平成一維，避免 stack 變 3D
mu = np.array([0.5, 3.0]); s2 = np.array([1.0, 1.0]); p = np.array([0.5, 0.5])
LL = []
for it in range(60):
    # E 步：軟歸屬（每個樣本對每個分量的機率）
    L = np.stack([gauss1(xx, mu[0], s2[0])*p[0], gauss1(xx, mu[1], s2[1])*p[1]], 1)   # (N,2)
    R = L / L.sum(1, keepdims=True)
    # M 步（向量化：每個分量獨立）
    Nk = R.sum(0)
    mu = (R.T @ xx) / Nk
    diff = xx[:, None] - mu[None, :]            # (N,2)
    s2 = ((diff ** 2) * R).sum(0) / Nk
    p = Nk / N
    LL.append(np.log(L.sum(1)).sum())
print('EM 收斂：μ ≈', np.round(mu, 3), ' σ² ≈', np.round(s2, 3), ' P ≈', np.round(p, 3))
plt.figure(figsize=(6, 3)); plt.plot(LL); plt.xlabel('迭代'); plt.ylabel('log-likelihood'); plt.title('EM：似然單調上升 ✅'); plt.show()

### ✏️ 練習
1. Parzen：維持 N=200，比較 h=0.05 vs 0.5 vs 2.0 的「偏差—方差」取捨；N=2000 時 h=0.1 是否變好？
2. EM：試著把兩分量改成「重疊多」的設定（μ=1 與 2），收斂值還分得開嗎？── 感受標籤未知的困難。